# Ejecutar MoneyPrinterTurbo en Google Colab

Este notebook instala [MoneyPrinterTurbo](https://github.com/harry0703/MoneyPrinterTurbo) en un entorno aislado de Python 3.11 y arranca su WebUI. Los entornos de ejecución de Colab son temporales, así que los archivos generados se eliminan cuando se reinicia el entorno.

## 1. Instalar MoneyPrinterTurbo

La instalación se puede volver a ejecutar sin problemas: la primera vez clona el repositorio y las siguientes lo actualiza. Las dependencias se instalan en el entorno virtual del proyecto, para no entrar en conflicto con los paquetes preinstalados de Colab.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/MoneyPrinterTurbo")
REPO_URL = "https://github.com/harry0703/MoneyPrinterTurbo.git"

# Actualiza el checkout existente para que volver a ejecutar esta celda no falle al clonar.
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True
    )
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} existe pero no es un repositorio Git")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True
    )

os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-q", "uv", "pyngrok"], check=True)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
# Usa el lockfile en un entorno aislado sin cambiar los paquetes preinstalados de Colab.
subprocess.run(["uv", "sync", "--frozen", "--python", "3.11"], check=True)
print(f"MoneyPrinterTurbo está listo en {REPO_DIR}")

## 2. Configurar el túnel de Colab

MoneyPrinterTurbo en sí no necesita ngrok para el uso local normal. Este notebook usa ngrok solo porque el servidor de Streamlit se ejecuta dentro de un entorno remoto de Colab.

Crea una cuenta y copia tu token de autenticación del [panel de ngrok](https://dashboard.ngrok.com/get-started/your-authtoken). La siguiente celda lo lee sin mostrarlo ni guardarlo en el notebook.

In [ ]:
from getpass import getpass

from pyngrok import ngrok

# Cierra los túneles de ejecuciones anteriores antes de configurar uno nuevo.
ngrok.kill()

ngrok_token = getpass("Introduce tu token de autenticación de ngrok: ").strip()
if not ngrok_token:
    raise ValueError("Hace falta un token de autenticación de ngrok")
ngrok.set_auth_token(ngrok_token)
del ngrok_token
print("Autenticación de ngrok configurada")

## 3. Arrancar la WebUI

La celda espera a que Streamlit esté sano antes de crear la URL pública. Si el arranque falla, incluye en el error el log reciente del servidor. Tras abrir la URL, usa **Ajustes** para configurar las claves API del modelo y de los medios que necesites. Para usar los LLM gratuitos de OpenRouter, elige **OpenRouter**, introduce tu clave API de OpenRouter y mantén el modelo por defecto `minimax/minimax-m3:free` o elige otro modelo de https://openrouter.ai/models?variant=free.

In [ ]:
import time
from pathlib import Path
import subprocess
from urllib.error import URLError
from urllib.request import urlopen

from pyngrok import ngrok

# Comprobaciones básicas antes de arrancar
REPO_DIR = Path("/content/MoneyPrinterTurbo")
if not REPO_DIR.is_dir():
    raise RuntimeError(f"El directorio del repositorio no existe: {REPO_DIR}")
if not (REPO_DIR / "webui" / "Main.py").is_file():
    raise RuntimeError(f"No se encontró webui/Main.py en {REPO_DIR}")

PORT = 8501
LOG_PATH = Path("/content/moneyprinterturbo-webui.log")

# Permite volver a ejecutar esta celda cerrando el proceso, el archivo de log y el túnel anteriores.
previous_streamlit_proc = globals().get("streamlit_proc")
if previous_streamlit_proc is not None and previous_streamlit_proc.poll() is None:
    previous_streamlit_proc.terminate()
    previous_streamlit_proc.wait(timeout=10)
previous_streamlit_log = globals().get("streamlit_log")
if previous_streamlit_log is not None and not previous_streamlit_log.closed:
    previous_streamlit_log.close()

try:
    ngrok.kill()
except Exception:
    pass

streamlit_log = LOG_PATH.open("w", encoding="utf-8")
streamlit_proc = subprocess.Popen(
    [
        "uv",
        "run",
        "streamlit",
        "run",
        "webui/Main.py",
        f"--server.port={PORT}",
        "--server.address=0.0.0.0",
        "--browser.gatherUsageStats=False",
        "--client.toolbarMode=minimal",
        "--server.showEmailPrompt=False",
    ],
    cwd=REPO_DIR,
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
    text=True,
)

# Sondea el endpoint de salud porque el tiempo de arranque varía entre entornos de Colab.
deadline = time.time() + 90
server_ready = False
while time.time() < deadline:
    if streamlit_proc.poll() is not None:
        break
    try:
        with urlopen(f"http://127.0.0.1:{PORT}/_stcore/health", timeout=2) as response:
            server_ready = response.status == 200
    except (URLError, TimeoutError):
        pass
    if server_ready:
        break
    time.sleep(2)

if not server_ready:
    streamlit_log.flush()
    recent_log = LOG_PATH.read_text(encoding="utf-8", errors="replace")[-4000:]
    raise RuntimeError(f"Streamlit failed to start. Recent log:\n{recent_log}")

# Usa una URL IPv4 explícita porque, si no, pyngrok puede enrutar localhost por ::1.
public_tunnel = ngrok.connect(addr=f"http://127.0.0.1:{PORT}", proto="http", bind_tls=True)
print("MoneyPrinterTurbo está listo:")
print(public_tunnel.public_url)
print(f"Log del servidor: {LOG_PATH}")